###  Análise exploratória dos dados Ler 
as tabelas do seu domínio usando python. Criar um relatório técnico no notebook respondendo: Tipos de dados, contagem de nulos, valores máximos/mínimos e chaves primárias.

In [0]:
import os
from dotenv import load_dotenv

load_dotenv()

jdbc_hostname = os.getenv("SQL_HOST")
jdbc_database = os.getenv("SQL_DATABASE")
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")

In [0]:
storage_account_name = "internshipdatalake"
container_name = "real-time-ecommerce-data"

print(storage_account_name)
print(container_name)

In [0]:
df_rastreamento = (
    spark.read
    .format("sqlserver")
    .option("host", jdbc_hostname)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .option("dbtable", "squad2.ecommerce_rastreamento")
    .option("encrypt", "true")
    .option("trustServerCertificate", "false")
    .load()
)

display(df_rastreamento)

In [0]:
#Tipos de dados
df_rastreamento.printSchema()

In [0]:
display(
    spark.createDataFrame(
        df_rastreamento.dtypes,
        ["coluna", "tipo_dado"]
    )
)

In [0]:
##Quantidade de registros e colunas
print(f"Total de registros: {df_rastreamento.count()}")
print(f"Total de colunas: {len(df_rastreamento.columns)}")

In [0]:
#Contagem de nulos por coluna
from pyspark.sql.functions import col, sum as spark_sum, when

df_nulos = df_rastreamento.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_rastreamento.columns
])

display(df_nulos)

In [0]:
#Valores mínimos e máximos
from pyspark.sql.functions import min as spark_min, max as spark_max

df_min_max = df_rastreamento.select(
    spark_min("id_rastreamento").alias("id_rastreamento_min"),
    spark_max("id_rastreamento").alias("id_rastreamento_max"),

    spark_min("id_pedido_ecommerce").alias("id_pedido_ecommerce_min"),
    spark_max("id_pedido_ecommerce").alias("id_pedido_ecommerce_max"),

    spark_min("id_transportadora").alias("id_transportadora_min"),
    spark_max("id_transportadora").alias("id_transportadora_max"),

    spark_min("dt_evento").alias("dt_evento_min"),
    spark_max("dt_evento").alias("dt_evento_max")
)

display(df_min_max)

In [0]:
#Verificar chave primária
total = df_rastreamento.count()
distintos = df_rastreamento.select("id_rastreamento").distinct().count()
nulos_pk = df_rastreamento.filter(col("id_rastreamento").isNull()).count()

print(f"Total de registros: {total}")
print(f"IDs distintos: {distintos}")
print(f"IDs nulos: {nulos_pk}")

if total == distintos and nulos_pk == 0:
    print("id_rastreamento pode ser considerada chave primária.")
else:
    print("id_rastreamento NÃO pode ser considerada chave primária.")

# Análise Exploratória - Tabela ecommerce_rastreamento

## Objetivo

Realizar uma análise exploratória da tabela `ecommerce_rastreamento`, identificando sua estrutura, tipos de dados, qualidade das informações e possíveis chaves primárias.

---

## Descrição da Tabela

A tabela `ecommerce_rastreamento` armazena eventos relacionados ao acompanhamento da entrega de pedidos do e-commerce. Seu objetivo é permitir o monitoramento do status logístico dos pedidos ao longo do processo de entrega.

---

## Estrutura dos Dados

A tabela é composta pelas seguintes colunas:

| Coluna | Tipo de Dado |
|----------|------------|
| id_rastreamento | long |
| id_pedido_ecommerce | long |
| codigo_rastreio | string |
| id_transportadora | long |
| status_entrega | string |
| dt_evento | timestamp |
| observacao | string |

---

## Análise de Valores Nulos

Foi realizada a contagem de valores nulos em todas as colunas da tabela com o objetivo de avaliar a qualidade dos dados e identificar possíveis inconsistências.

Os resultados obtidos podem ser visualizados na tabela gerada pela célula de análise de nulos.

---

## Valores Mínimos e Máximos

Foram analisadas as colunas numéricas e temporais da tabela para identificar os limites dos dados armazenados.

### Colunas analisadas

- id_rastreamento
- id_pedido_ecommerce
- id_transportadora
- dt_evento

Os valores mínimos e máximos encontrados são apresentados na tabela gerada pelo notebook.

---

## Análise da Chave Primária

A coluna `id_rastreamento` foi avaliada como candidata à chave primária da tabela.

A validação foi realizada comparando:

- Quantidade total de registros;
- Quantidade de valores distintos da coluna;
- Existência de valores nulos.

Caso o total de registros seja igual ao total de valores distintos e não existam valores nulos, a coluna pode ser considerada uma chave primária válida.

---

## Considerações sobre o Status de Entrega

Durante a análise foi identificado apenas um valor distinto para a coluna `status_entrega`:

- Aguardando Coleta

Isso indica que os registros presentes representam apenas a etapa inicial do fluxo logístico, não sendo observados eventos de transporte ou entrega concluída no conjunto de dados analisado.

---

## Conclusão

A tabela `ecommerce_rastreamento` apresenta estrutura consistente para análises logísticas e de monitoramento de entregas.

Foram avaliados os tipos de dados, a presença de valores nulos, os valores mínimos e máximos das colunas relevantes e a possível chave primária da tabela.

Os dados encontram-se estruturados e adequados para utilização em análises operacionais e futuras etapas do projeto de Engenharia de Dados.